In [1]:
import os
import sys
import importlib
import math
import time

import itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from lion_pytorch import Lion
import pytorch_lightning as pl

from IPython.display import clear_output

In [2]:
sys.path.append("../../model")
import utrdata_cl as utrdata
from legnet import LegNetClassifier
from pl_regressor import RNARegressor

In [3]:
UTR_TYPE = "utr5"
SEQSIZE = 50 if UTR_TYPE == "utr5" else 240
DEVICE = 0 if UTR_TYPE == "utr5" else 1

## Loading data

In [4]:
from ablation_utils import load_data, launch_model

In [5]:
splits = load_data(utr_type=UTR_TYPE, prefix="..")

batch_size = 1024
steps_per_epoch = max(1, splits["train"].shape[0] // batch_size)
num_workers = 32

In [6]:
def parameter_ablation(
    model_name: str,
    parameter_update: dict,
):
    checked = {
        "seeds": [0, 1, 2, 3, 4],
        "features": ("sequence", "positional", "conditions"),  # ("sequence", "positional", "conditions"),
        "augment_dict": dict(
            extend_left=0,
            extend_right=0,
            shift_left=0,
            shift_right=0,
            revcomp=False,
        ),
        "epochs": 10,
        "predict_cols": ['mass_center'],
        "kernel_size": 3,
        "conv_sizes": (128, 64, 64, 32, 32),
        "optimizer": (
            torch.optim.AdamW,
            dict(
                lr=0.01,
                weight_decay=0.1,
            ),
        ),
        "scheduler": (
            torch.optim.lr_scheduler.OneCycleLR,
            dict(
                max_lr=0.01,
                steps_per_epoch=steps_per_epoch,
                epochs=None,
                pct_start=0.3,
                three_phase=False,
                cycle_momentum=True,
            ),
        ),
    }
    checked.update(parameter_update)
    if 'epochs' in checked['scheduler'][1]:
        checked['scheduler'][1]['epochs'] = checked['epochs']

    PARAMS = checked.copy()
    AUGMENT_KEY = any(PARAMS["augment_dict"].values())
    AUGMENT_TEST_TIME = AUGMENT_KEY

    METRICS = []
    for SEED in PARAMS["seeds"]:
        metrics = launch_model(
            model_name=model_name,
            utr_type=UTR_TYPE,
            splits=splits,
            seed=SEED,
            batch_size=batch_size,
            train_ds_kws=dict(
                predict_cols=PARAMS["predict_cols"],
                construct_type=UTR_TYPE,
                features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
                augment=AUGMENT_KEY,
                augment_test_time=False,
                augment_kws=PARAMS["augment_dict"],
            ),
            val_ds_kws=dict(
                predict_cols=PARAMS["predict_cols"],
                construct_type=UTR_TYPE,
                features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
                augment=False,
                augment_test_time=AUGMENT_TEST_TIME,
                augment_kws=PARAMS["augment_dict"],
            ),
            model_class=LegNetClassifier,
            model_kws=dict(
                seqsize=SEQSIZE,
                ks=PARAMS["kernel_size"],
                out_channels=PARAMS["predict_cols"].__len__(),
                conv_sizes=PARAMS["conv_sizes"],
                mapper_size=256,
                linear_sizes=None,
                use_max_pooling=False,
                final_activation=nn.Identity
            ),
            criterion_class=nn.MSELoss,
            criterion_kws=dict(),
            optimizer_class=PARAMS['optimizer'][0],
            optimizer_kws=PARAMS['optimizer'][1],
            lr_scheduler_class=PARAMS['scheduler'][0],
            lr_scheduler_kws=PARAMS['scheduler'][1],
            test_time_validation=AUGMENT_TEST_TIME,
            epochs=PARAMS["epochs"],
            num_workers=num_workers,
            device=DEVICE,
        )
        METRICS.append(metrics)
    return METRICS

## Launching ablation

In [ ]:
ablation_tests = {
    "blocks=5_ReduceLR": {
        "epochs": 50,
        "optimizer": (
            torch.optim.AdamW,
            dict(
                lr=0.01,
                weight_decay=0.1,
            )
        ),
        "scheduler": (
            torch.optim.lr_scheduler.ReduceLROnPlateau,
            dict(
                factor=0.5,
                patience=84 if UTR_TYPE == "utr5" else 133,
                cooldown=10,
            )
        ),
    },
    "blocks=5_NoScheduler": {
        "epochs": 50,
        "optimizer": (
            torch.optim.AdamW,
            dict(
                lr=0.01,
                weight_decay=0.1,
            )
        ),
        "scheduler": (None, dict()),
    },
    "blocks=5_Lion_NoScheduler": {
        "epochs": 50,
        "optimizer": (
            Lion,
            dict(
                lr=0.01,
                weight_decay=0.1,
            )
        ),
        "scheduler": (None, dict()),
    },
    "blocks=5_Lion_lr=1e-2_wd=0.1": {
        "optimizer": (
            Lion,
            dict(
                weight_decay=0.1,
            )
        ),
        "scheduler": (
            torch.optim.lr_scheduler.OneCycleLR,
            dict(
                max_lr=1e-2,
                steps_per_epoch=steps_per_epoch,
                epochs=None,
                pct_start=0.3,
                three_phase=False,
                cycle_momentum=True,
            ),
        ),
    },
    "blocks=5_Lion_lr=1e-3_wd=0.1": {
        "optimizer": (
            Lion,
            dict(
                weight_decay=0.1,
            )
        ),
        "scheduler": (
            torch.optim.lr_scheduler.OneCycleLR,
            dict(
                max_lr=1e-3,
                steps_per_epoch=steps_per_epoch,
                epochs=None,
                pct_start=0.3,
                three_phase=False,
                cycle_momentum=True,
            ),
        ),
    },
    "blocks=5_ks=5": {
        "kernel_size": 5,
    },
    "blocks=5_ks=7": {
        "kernel_size": 7,
    },
    "blocks=5_doubleconv": {
        "conv_sizes": (256, 128, 128, 64, 64),
    },
    "blocks=3": {
        "conv_sizes": (128, 64, 32),
    },
    "blocks=4": {
        "conv_sizes": (128, 64, 64, 32),
    },
    "blocks=5": {
        "conv_sizes": (128, 64, 64, 32, 32),
    },
    "blocks=6": {
        "conv_sizes": (128, 64, 64, 32, 32, 16),
    },
    "blocks=7": {
        "conv_sizes": (128, 64, 64, 32, 32, 16, 16),
    },
    "blocks=5_remcodon": {
        "features": ("sequence", "conditions"),
        "predict_cols": ["mass_center"],
    },
    "blocks=5_dualreg_remcodon": {
        "features": ("sequence", "conditions"),
        "predict_cols": ["diff", "mass_center"],
    },
    "blocks=5_dualreg": {
        "predict_cols": ["diff", "mass_center"],  # mass_center must be last
    },
}

ablation_metrics = list()
for test_name, update_dict in ablation_tests.items():
    iter_metrics = parameter_ablation(model_name=f"{UTR_TYPE}_{test_name}", parameter_update=update_dict)
    ablation_metrics.extend(iter_metrics)

Global seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type             | Params
-----------------------------------------------
0 | model     | LegNetClassifier | 267 K 
1 | criterion | MSELoss          | 0     
-----------------------------------------------
267 K     Trainable params
0         Non-trainable params
267 K     Total params
1.071     Total estimated model params size (MB)
Traceback (most recent call last):
  File "/home/arsen_l/.minicond

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
import json

with open(f"{UTR_TYPE}_ablation_stepwise.joint.json", "wt") as handle:
    json.dump(ablation_metrics, handle)

---